In [1]:
%pip install httpx edgar-sec orjson

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime

START_YEAR = datetime.strptime("2011", "%Y").year
CHANGED_FORMAT_YEAR = datetime.strptime("2022", "%Y").year
END_YEAR = datetime.today().year

append_date_template = f"-01-31"
file_extension = ".zip"


def validate_target_year(target_year: int) -> None:
  if target_year not in range(START_YEAR, END_YEAR):
    raise ValueError(f"Target year must be between {START_YEAR} and {END_YEAR}")
  return None

def target_url(target_year: int) -> str:
  validate_target_year(target_year)
  xbrl_url_template = f"https://xbrl.fasb.org/us-gaap/{target_year}/us-gaap-{target_year}"
  if target_year < CHANGED_FORMAT_YEAR:
    return xbrl_url_template + append_date_template + file_extension
  return xbrl_url_template + file_extension

In [3]:
print(target_url(2011))

https://xbrl.fasb.org/us-gaap/2011/us-gaap-2011-01-31.zip


In [4]:
import httpx

client: httpx.Client = httpx.Client()

HEADERS: dict[str, str] = {
    "User-Agent": "fedfred/edgar-sec Nikhil Sunder nsunder724@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}

resp = client.get(url = target_url(2022), headers = HEADERS)


print(resp)


<Response [200 OK]>


In [5]:
from collections.abc import Mapping

def pull_file_range(start_year: int, end_year: int) -> list[Mapping]:
  for year in range(start_year, end_year + 1):
    url = target_url(year)
    response = client.get(url, headers = HEADERS)
    response.raise_for_status()
    print(response)


In [6]:
print(pull_file_range(2011, 2022))

<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
<Response [200 OK]>
None


In [7]:
import httpx
from pathlib import Path

def fetch_taxonomy(year: int, dest: Path) -> Path:
    url = target_url(year)
    out = Path(dest) / f"us-gaap-{year}.zip"
    with httpx.stream("GET", url, follow_redirects=True, timeout=60.0, headers=HEADERS) as r:
        r.raise_for_status()
        with out.open("wb") as f:
            for chunk in r.iter_bytes():
                f.write(chunk)
    return out

fetch_taxonomy(2023, "")

PosixPath('us-gaap-2023.zip')